# Object detection throght VDO Chapter 11 Workshop 2

## Load module

In [1]:
import cv2
import numpy as np

## Setting variable

In [2]:
colorGreen = (0,255,0)          # Green color
scale = 0.30                    # Scale for resizing the image
img_size = (608, 608)           # Size of the input image for the model
confThreshold = 0.5             # Confidence threshold for filtering detections
nmsThreshold = 0.4              # Non-Maximum Suppression threshold

# Usecase camera
# cap = cv2.VideoCapture(0)       # Open the default camera
# cap.set(3, 640)                 # Set width
# cap.set(4, 480)                 # Set height

# Usecase video file
source_path = "../datasets/video/car_on_road.mp4"
cap = cv2.VideoCapture(source_path)  # Open a video file

if not cap.isOpened():          # case video file or camera not opened
    print("Error: Could not open video.")
    # exit()

# weight_path = "D:\Project\Learning\TensorFlow_Files\model\yolov3\yolov3.weights"
# cfg_path = "D:\Project\Learning\TensorFlow_Files\model\yolov3\yolov3.cfg"
# coco_names_path = "D:\Project\Learning\TensorFlow_Files\model\yolov3\coco.names"

weight_path = "../models/yolov3/yolov3.weights"
cfg_path = "../models/yolov3/yolov3.cfg"
coco_names_path = "../models/yolov3/coco.names"






Error: Could not open video.


## Load class name from .name file

In [4]:
classeName = []
# Load class names from coco.names file
with open(coco_names_path, 'r') as f:
    classeName = [line.strip() for line in f.readlines()]


## Load YOLO model

In [5]:
# Load the YOLO model
net = cv2.dnn.readNetFromDarknet(cfg_path, weight_path)
layerNames = net.getLayerNames()
# Get the output layer names
outputLayers = [layerNames[i - 1] for i in net.getUnconnectedOutLayers()]


## Declare object rectangular function

In [6]:
def locateObject(outputs, img):
    hT, wT, cT = img.shape
    bbox = []
    classIds = []
    confs = []

    for output in outputs:
        for detection in output:
            scores = detection[5:]
            classId = np.argmax(scores)
            confidence = scores[classId]
            if confidence > confThreshold:
                w, h = int(detection[2] * wT), int(detection[3] * hT)
                x, y = int((detection[0] * wT) - w / 2), int((detection[1] * hT) - h / 2)

                bbox.append([x, y, w, h])
                classIds.append(classId)
                confs.append(float(confidence))
    # Apply Non-Maximum Suppression to remove overlapping boxes
    indices = cv2.dnn.NMSBoxes(bbox, confs, confThreshold, nmsThreshold)
    
    # Return the bounding boxes, class IDs, and confidences of detected objects
    for i in indices:
        # i = i[0]  # Convert to scalar
        box = bbox[i]
        x, y, w, h = box # box[0], box[1], box[2], box[3]
        cv2.rectangle(img, (x, y), (x + w, y + h), colorGreen, 2)
        cv2.putText(img, f"{classeName[classIds[i]]} {int(confs[i] * 100)}%", 
                    (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colorGreen, 2)

## Main Loop

In [ ]:
while (cap.isOpened()):             # Loop for reading frames from the video or camera
    ret, img = cap.read()           # img is the current frame
    if not ret:
        print("Error: Could not read frame.")
        # cv2.waitKey(3000)            # Wait for 3 seconds before exiting
        break  # Exit the loop if no frame is captured

    # Resize the image to the required size
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
    # Create a blob from the image
    blob = cv2.dnn.blobFromImage(img, 1/255.0, img_size, (0, 0, 0), swapRB=True, crop=False)
    
    # Set the input to the network
    net.setInput(blob)
    
    # Forward pass through the network
    outputs = net.forward(outputLayers)
    
    # Locate objects in the outputs
    locateObject(outputs, img)
    
    # Display the image with detections
    cv2.imshow("Object Detection", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break  # Exit on 'q' key press

# Release the video capture object and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()

# Object detection through laptop camera Chapter 11 Workshop 3

## change setting variable for using laptop camera

In [7]:
# Usecase camera
cap = cv2.VideoCapture(0)       # Open the default camera
cap.set(3, 640)                 # Set width
cap.set(4, 480)                 # Set height

True

## run same while from workshop 2

In [8]:
while (cap.isOpened()):             # Loop for reading frames from the video or camera
    ret, img = cap.read()           # img is the current frame
    if not ret:
        print("Error: Could not read frame.")
        # cv2.waitKey(3000)            # Wait for 3 seconds before exiting
        break  # Exit the loop if no frame is captured

    # Resize the image to the required size
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
    # Create a blob from the image
    blob = cv2.dnn.blobFromImage(img, 1/255.0, img_size, (0, 0, 0), swapRB=True, crop=False)
    
    # Set the input to the network
    net.setInput(blob)
    
    # Forward pass through the network
    outputs = net.forward(outputLayers)
    
    # Locate objects in the outputs
    locateObject(outputs, img)
    
    # Display the image with detections
    cv2.imshow("Object Detection", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break  # Exit on 'q' key press

# Release the video capture object and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()